## Preprocessing

In [1]:
# Import our dependencies
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import tensorflow as tf

# Import pandas and read the charity_data.csv from the provided cloud URL.
import pandas as pd
application_df = pd.read_csv("https://static.bc-edx.com/data/dl-1-2/m21/lms/starter/charity_data.csv")
application_df.head()

,EIN,NAME,APPLICATION_TYPE,AFFILIATION,CLASSIFICATION,USE_CASE,ORGANIZATION,STATUS,INCOME_AMT,SPECIAL_CONSIDERATIONS,ASK_AMT,IS_SUCCESSFUL
0,10520599,BLUE KNIGHTS MOTORCYCLE CLUB,T10,Independent,C1000,ProductDev,Association,1,0,N,5000,1
1,10531628,AMERICAN CHESAPEAKE CLUB CHARITABLE TR,T3,Independent,C2000,Preservation,Co-operative,1,1-9999,N,108590,1
2,10547893,ST CLOUD PROFESSIONAL FIREFIGHTERS,T5,CompanySponsored,C3000,ProductDev,Association,1,0,N,5000,0
3,10553066,SOUTHSIDE ATHLETIC ASSOCIATION,T3,CompanySponsored,C2000,Preservation,Trust,1,10000-24999,N,6692,1
4,10556103,GENETIC RESEARCH INSTITUTE OF THE DESERT,T3,Independent,C1000,Heathcare,Trust,1,100000-499999,N,142590,1


In [2]:
# Drop the non-beneficial ID columns, 'EIN' and 'NAME'.
application_df = application_df.drop(columns=["EIN", "NAME"])

In [3]:
# Determine the number of unique values in each column.
application_df.nunique()

,0
APPLICATION_TYPE,17
AFFILIATION,6
CLASSIFICATION,71
USE_CASE,5
ORGANIZATION,4
STATUS,2
INCOME_AMT,9
SPECIAL_CONSIDERATIONS,2
ASK_AMT,8747
IS_SUCCESSFUL,2


In [4]:
# Look at APPLICATION_TYPE value counts to identify and replace with "Other"
application_df["APPLICATION_TYPE"].value_counts()

,count
APPLICATION_TYPE,
T3,27037
T4,1542
T6,1216
T5,1173
T19,1065
T8,737
T7,725
T10,528
T9,156


In [5]:
# Choose a cutoff value and create a list of application types to be replaced
application_type_counts = application_df["APPLICATION_TYPE"].value_counts()
application_types_to_replace = application_type_counts[application_type_counts < 500].index.tolist()

# Replace in dataframe
for app in application_types_to_replace:
    application_df['APPLICATION_TYPE'] = application_df['APPLICATION_TYPE'].replace(app, "Other")

# Check to make sure replacement was successful
application_df['APPLICATION_TYPE'].value_counts()


,count
APPLICATION_TYPE,
T3,27037
T4,1542
T6,1216
T5,1173
T19,1065
T8,737
T7,725
T10,528
Other,276


In [6]:
# Look at CLASSIFICATION value counts
classification_counts = application_df["CLASSIFICATION"].value_counts()

# Choose a cutoff value and create a list of classification types to be replaced
classification_types_to_replace = classification_counts[classification_counts < 500].index.tolist()

# Replace in dataframe
for cls in classification_types_to_replace:
    application_df['CLASSIFICATION'] = application_df['CLASSIFICATION'].replace(cls, "Other")

# Check to make sure replacement was successful
application_df['CLASSIFICATION'].value_counts()


,count
CLASSIFICATION,
C1000,17326
C2000,6074
C1200,4837
C3000,1918
C2100,1883
Other,1484
C7000,777


In [7]:
# Look at CLASSIFICATION value counts greater than 1
classification_counts = application_df["CLASSIFICATION"].value_counts()

# Filter for counts greater than 1
classification_counts_above_1 = classification_counts[classification_counts > 1]

# Display the results
print(classification_counts_above_1)


CLASSIFICATION
C1000    17326
C2000     6074
C1200     4837
C3000     1918
C2100     1883
Other     1484
C7000      777
Name: count, dtype: int64


In [8]:
# Choose a cutoff value, for example, classifications with fewer than 100 occurrences
cutoff_value = 100

# Create a list of classifications to be replaced
classifications_to_replace = application_df['CLASSIFICATION'].value_counts()[application_df['CLASSIFICATION'].value_counts() < cutoff_value].index.tolist()

# Replace in dataframe
for cls in classifications_to_replace:
    application_df['CLASSIFICATION'] = application_df['CLASSIFICATION'].replace(cls, "Other")

# Check to make sure replacement was successful
application_df['CLASSIFICATION'].value_counts()


,count
CLASSIFICATION,
C1000,17326
C2000,6074
C1200,4837
C3000,1918
C2100,1883
Other,1484
C7000,777


In [9]:
# Convert categorical data to numeric using pd.get_dummies
application_df = pd.get_dummies(application_df, drop_first=True)


In [10]:
# Split our preprocessed data into our features and target arrays
X = application_df.drop(columns=["IS_SUCCESSFUL"])  # Features
y = application_df["IS_SUCCESSFUL"]  # Target variable

# Split the preprocessed data into a training and testing dataset
from sklearn.model_selection import train_test_split

# Split the data into training and testing datasets (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
# Create a StandardScaler instances
scaler = StandardScaler()

# Fit the StandardScaler
X_scaler = scaler.fit(X_train)

# Scale the data
X_train_scaled = X_scaler.transform(X_train)
X_test_scaled = X_scaler.transform(X_test)

## Compile, Train and Evaluate the Model

In [13]:
from keras.models import Sequential
from keras.layers import Dense

# Define the model - deep neural net
nn = tf.keras.models.Sequential()

# First hidden layer
# Assuming the input features are X_train_scaled with shape (n_samples, n_features)
nn.add(tf.keras.layers.Dense(units=128, activation='relu', input_dim=X_train_scaled.shape[1]))

# Second hidden layer
nn.add(tf.keras.layers.Dense(units=64, activation='relu'))

# Output layer
# For binary classification, we use 1 unit with a sigmoid activation function
nn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

# Check the structure of the model
nn.summary()


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         4,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,185 (51.50 KB)

 Trainable params: 13,185 (51.50 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
# Compile the model
nn.compile(loss='binary_crossentropy',   # Loss function for binary classification
           optimizer='adam',             # Optimizer (Adam is commonly used)
           metrics=['accuracy'])         # Metrics to evaluate model performance


In [15]:
# Train the model
history = nn.fit(X_train_scaled, y_train,
                 epochs=100,         # Number of epochs
                 batch_size=32,      # Batch size
                 validation_data=(X_test_scaled, y_test),  # Validation data
                 verbose=2)          # Verbose output (level 2 shows a progress bar)


Epoch 1/100
858/858 - 4s - 5ms/step - accuracy: 0.7223 - loss: 0.5670 - val_accuracy: 0.7268 - val_loss: 0.5603
Epoch 2/100
858/858 - 3s - 3ms/step - accuracy: 0.7299 - loss: 0.5553 - val_accuracy: 0.7259 - val_loss: 0.5613
Epoch 3/100
858/858 - 3s - 4ms/step - accuracy: 0.7293 - loss: 0.5513 - val_accuracy: 0.7289 - val_loss: 0.5616
Epoch 4/100
858/858 - 4s - 5ms/step - accuracy: 0.7312 - loss: 0.5501 - val_accuracy: 0.7265 - val_loss: 0.5583
Epoch 5/100
858/858 - 3s - 3ms/step - accuracy: 0.7317 - loss: 0.5483 - val_accuracy: 0.7229 - val_loss: 0.5580
Epoch 6/100
858/858 - 3s - 3ms/step - accuracy: 0.7305 - loss: 0.5472 - val_accuracy: 0.7267 - val_loss: 0.5586
Epoch 7/100
858/858 - 5s - 5ms/step - accuracy: 0.7317 - loss: 0.5467 - val_accuracy: 0.7259 - val_loss: 0.5586
Epoch 8/100
858/858 - 2s - 3ms/step - accuracy: 0.7321 - loss: 0.5468 - val_accuracy: 0.7249 - val_loss: 0.5561
Epoch 9/100
858/858 - 3s - 3ms/step - accuracy: 0.7322 - loss: 0.5458 - val_accuracy: 0.7233 - val_loss:

In [16]:
# Evaluate the model using the test data
model_loss, model_accuracy = nn.evaluate(X_test_scaled,y_test,verbose=2)
print(f"Loss: {model_loss}, Accuracy: {model_accuracy}")

215/215 - 1s - 3ms/step - accuracy: 0.7268 - loss: 0.5661
Loss: 0.5660607814788818, Accuracy: 0.7268221378326416


In [17]:
# Export our model to HDF5 file
nn.save("AlphabetSoupCharity_Optimization.h5")